## TIFF plot

In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png")

# install.packages("ncdf4", "tidyverse", "terra", "dplyr", "sf", "jsonlite", "utils")
# install.packages("ncdf4")

# GET THE DEFAULT INITIAL PARAMETERS

## CONDA ENVIRONEMENT (OPTIONAL)
Sys.getenv("CONDA_DEFAULT_ENV")
## VERSION OF R IN USE
version
# getwd()
# list.files("data/2008")

# setwd("/path/to/your/folder")


system("conda activate")

In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse)
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)
# ---
library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"data",sep = "/")
outputDir <- paste(workdir,"outputs",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.file <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
        } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename){
    # nchar(filename)
    filename.length <- nchar(filename)
    
    # Get the extension ("." + 2 letters)
    # substr(filename,filename.length-2, filename.length)
    
    # Get the name without the extension ("." + 2 letters)
    start <- 1
    end <- filename.length-3
    unzip.filename <- substr(filename,start, end)

    
    if(!file.exists(unzip.filename)){
 
        # print(global_topo_tiff_gz)
        # print(global_topo_tiff)
        
        R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}

---
# Break

In [ ]:
global_topo_tiff_gz <- "global_topo.tiff.gz"

# ==============================================================================
# DOWNLOAD FILENAME
bathy.url <- "https://topex.ucsd.edu/pub/global_topo_tiff/global.tiff.gz"
download.file(global_topo_tiff_gz, bathy.url)

# ==============================================================================
# UNZIP FILENAME
global_topo_tiff <- unzip.file(global_topo_tiff_gz)


In [ ]:
# ==============================================================================
# FILE INFO
# ==============================================================================
print(file.info( global_topo_tiff_gz ))
print(file.info( (global_topo_tiff )))

ls()

In [ ]:
# MOVE TO DATA DIRECTORY 
global_topo_tiff_gz <- move.file(global_topo_tiff_gz, dataDir)
global_topo_tiff <- move.file(global_topo_tiff, dataDir)

## NetCDF (Copernicus Data Files)

In [ ]:
# 2.2 DOWNLOAD AND LOAD BATHYMETRY ----
bathy_file <- "global_topo_1min_topo_19_1.nc"
bathy.url <- "https://topex.ucsd.edu/pub/global_topo_1min/topo_19.1.nc"
download.file(bathy_file, bathy.url)

ls()

In [ ]:
# MOVE TO DATA DIRECTORY 
bathy_file <- move.file(bathy_file, dataDir)

# Ice concentration

In [ ]:
# 2.2 DOWNLOAD COPERNICUS NetCDF ----

# Sea ice area & Sea ice concentration estimates as retrieved by the algorithm,
# and that were edited away by the various filters
# system("python API_Copernicus.py")

# OR

# Use 'Copernicus marine data' output as an input
# ice_netcdf <- "./data/osisaf_obs-si_glo_phy_sic-south_my_amsr_cdr_P1D-m_1778859355280.nc"
ice_netcdf <- 'galaxy_inputs/Datasets/Copernicus marine data.netcdf'
nc_iceC_file <- nc_open(ice_netcdf)
print(nc_iceC_file)

In [ ]:
# Structure of the file

names(nc_iceC_file)
# Names of the variables
names(nc_iceC_file$var)
# Names of the dimensions
names(nc_iceC_file$dim)

In [ ]:
dim(dim_ice)
length(as.vector(dim_ice))
length(dim_ice_lon)
length(dim_ice_lat)
length(dim_ice_lon) * length(dim_ice_lat)

In [ ]:
#  Extract the coordinates
dim_ice_lon <- ncvar_get(nc_iceC_file, "longitude")
dim_ice_lat <- ncvar_get(nc_iceC_file, "latitude")
dim_ice_time <- ncvar_get(nc_iceC_file, "time")
dim_ice <- ncvar_get(nc_iceC_file, "ice_conc")
dim_raw_ice <- ncvar_get(nc_iceC_file, "raw_ice_conc_values")

names(nc_iceC_file$var$ice_conc)

t_units <- ncatt_get(nc_iceC_file, "time", "units")

#sea_water_potential_concentration
T <- c("ice_conc", "raw_ice_conc_values")

In [ ]:
dim(dim_ice_lon)
dim(dim_ice_lat)
dim(dim_ice_time)
dim(dim_ice)
dim(dim_raw_ice)

dim_ice_lon[1:5]
dim_ice_lat[1:5]
dim_ice_time[1:5]
dim_ice[1:5]
dim_raw_ice[1:5]

In [ ]:
# convert time -- split the time units string into fields
t_ustr <- strsplit(t_units$value, " ")
t_dstr <- strsplit(unlist(t_ustr)[3], "-")
date <- ymd(t_dstr) + dseconds(dim_ice_time)
# date[1:5]

In [ ]:
T_array_raw_ice <- dim_raw_ice
T_array_ice <- dim_ice

# Quick map plot

# set the time step
t <- 1 #temperature on 2003-01-01
T_slice <- T_array_ice[,,t]


In [ ]:
image(longitude,latitude,T_slice, col = rev(brewer.pal(10,"RdBu")))

In [ ]:
grid <- expand.grid(lon=longitude, lat=latitude)  #create a set of lonxlat pairs of values, one for each element in the Temp_array

cutpts <- c(12,13,14,15,16,17,18,19,20)   # set colorbar
levelplot(T_slice ~ lon * lat,
          data=grid, region=TRUE,
          pretty=T, at=cutpts, cuts=9,
          col.regions=(rev(brewer.pal(9,"RdBu"))), contour=0,
          xlab = "Longitude", ylab = "Latitude",
          main = "Sea Water Potential Temperature (°C)"
          )

In [ ]:
ice_coords <- expand.grid(dim_ice_lon, dim_ice_lat,dim_ice,date)
ice_matrix <- cbind(ice_coords, as.vector(dim_ice))
names(ice_matrix) <- c("lon", "lat", "iceC")
head(ice_matrix)

In [ ]:
dim_ice_lon[1:5]
dim_ice_lat[1:5]
dim_ice[1:5]

In [ ]:
# summary(dim_sst)
sum(is.na(dim_ice))
all(is.na(dim_ice))
# dim_ice <- ncvar_get(nc_iceC_file, "iceC", raw_datavals = TRUE)
# range(dim_ice)

In [ ]:
ice_raw <- ncvar_get(nc_iceC_file, "iceC", raw_datavals = TRUE)

# Get attributes
fill_value <- ncatt_get(nc_iceC_file, "iceC", "_FillValue")$value ; fill_value
scale      <- ncatt_get(nc_iceC_file, "iceC", "scale_factor")$value ; scale

ice_raw[ice_raw == fill_value] <- NA
ice <- ice * scale
# Fill Missing values by replacing NA with the lowest values
range(dim_ice, na.rm = TRUE)

In [ ]:
nc_close(nc_iceC_file)

## Data Exploration

File exploration : ornldaac [github repository](https://github.com/ornldaac/netCDF_data_in_R/blob/master/netCDF_in_r_ornldaac_tutorial.md)

In [ ]:
## A.7 Draw Pairs Plot of Data Frame Columns
'install.packages("GGally")             # Install GGally package
library("GGally")                      # Load GGally package'

# ggpairs(Bathy)                        # Draw pairs plot

In [ ]:
## A.8 Boxplots of Multiple Columns 
# ggplot(as.data.frame(Bathy),                    # Draw boxplots
#        aes(x = value,
#            fill = name)) +
#   geom_boxplot()

In [ ]:
## A.9 Histograms of Multiple Columns 
# ggplot(Bathy,                    # Draw histograms
#        aes(x = value)) +
#   geom_histogram() + 
#   facet_wrap(name ~ ., scales = "free")

In [ ]:
cat("Content of", getwd(), ":\n", list.files(), "\n")
cat(bathy_file,"exists:", file.exists(bathy_file),"\n")
# Check for the file
cat("Size:", file.info(bathy_file)$size)
 # If size is 0 or very small, the file is broken.

In [ ]:
# Open the NetCDF file

nc_file <- nc_open(bathy_file)
print(nc_file)

# Structure of the file

names(nc_file)
# Names of the variables
names(nc_file$var)
# Names of the dimensions
names(nc_file$dim)

In [ ]:
# List the attributes and sub-attributes

i <- 1
for(listVar in names(nc_file)){
    cat(i, listVar,"\n")
    for(listNames in names(nc_file[[listVar]])){
        cat("Attr:", listNames, ":", names(nc_file[[listVar]][[listNames]]),"\n")
    }
    i <- i + 1
}

In [ ]:
# Sub-Content for lat and lon
names(nc_file$dim$lon)
names(nc_file$dim$lat)

nc_file$dim$lon[1:5]
nc_file$dim$lat[1:5]

# 5 first Values
nc_file$dim$lon$vals[1:5] # print(nc_file$dim['lon'])
nc_file$dim$lat$vals[1:5]
nc_file$var$z$id[1:5]


In [ ]:
# Get coordinates variables
longitude <- ncvar_get(nc_file,"lon")
latitude <- ncvar_get(nc_file,"lat")
z <- ncvar_get(nc_file,"z")

In [ ]:
# Dimensions of latitude & longitude
print(c(length(longitude), length(latitude)))
print(dim(z))

# Check longtitude and latitude values
cat(" Head of longitude:",head(longitude),"\n")
cat(" Head of latitude:",head(latitude),"\n")

fillvalue <- ncatt_get(nc_file, "z", "_FillValue") # The fill value (aka, the no data value) is -9999.
print(fillvalue$value)

In [ ]:
nc_close(nc_file)

In [ ]:
z[z == fillvalue$value] <- NA
nc.slice.min80 <- z[,1]
dim(nc.slice.min80)

In [ ]:
#r <- rast(nc.slice.min80,
#  extent = ext(min(lon), max(lon), min(lat), max(lat)),
#  crs = "+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs +towgs84=0,0,0"
#)
#rm(nc.slice.min80)

### Plots

In [ ]:
Bathy <- rast(bathy_file)
print("Converted into Raster File")

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
png <- 1
if (png) {
  png(filename = "outputs/Fig1.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/Fig1.pdf")
}

Depth_cuts <- c(-8200 ,-7000 ,-6000 ,-5000, -4000, -3000, -1800, -1400, -1000,  -600,  -400 , -200  ,   0 ,   50  , 250   ,500)
Depth_cols <- c(
  "#D6EAF8", "#AED6F1", "#85C1E9", "#5DADE2", "#3498DB",
  "#5DADE2", "#85C1E9", "#A9CCE3", "#D4E6F1", "#EBF5FB",
  "#F4F6F7", "#F8F9F9", "#FDFEFE", "#F2F3F4", "#EAEDED"
)

# Plot bathymetry
plot(Bathy,breaks = Depth_cuts, col = Depth_cols, legend = FALSE, axes = FALSE, box = FALSE,mar=c(0,0,0,0))

# Save the plot ---
dev.off()


### Convert NetCDF to CSV
Based on [Copernicus Marine Services](https://help.marine.copernicus.eu/en/articles/6328012-how-to-convert-netcdf-to-csv-using-r)

In [ ]:
#  Extract the coordinates
nc_file <- nc_open(bathy_file)

dim_lon <- ncvar_get(nc_file, "lon", collapse_degen=FALSE)
dim_lat <- ncvar_get(nc_file, "lat", collapse_degen=FALSE)
dim_depth <- ncvar_get(nc_file, "z", collapse_degen=FALSE)

In [ ]:
dim_lon[1:5]
dim_lat[1:5]
dim_depth[1:5]

In [ ]:

check_dim <- function(dim_var){
    cat(str(dim_var),
    length(dim_var),
    any(is.na(dim_var)),sep="\n")
}

check_dim(dim_lon)
print("---*---")
check_dim(dim_lat)
print("---*---")
check_dim(dim_depth)

In [ ]:
dim(dim_depth)

In [ ]:
coords <- expand.grid(dim_lon, dim_lat)
depth_matrix <- data.frame(cbind(coords, as.vector(dim_depth)))

In [ ]:
names(depth_matrix) <- c("lon", "lat", "depth")
head(depth_matrix)
# head(df.depth)
nc_close(nc_file)

In [ ]:
output.directory <- "outputs"
ifelse(!dir.exists(file.path(output.directory)),
        dir.create(file.path(output.directory)),
        "Directory Exists")

head(na.omit(depth_matrix), 5)  # Display some non-NaN values for a visual check
csv_fname <- paste(output.directory,"netcdf_depth.csv", sep="/")
write.table(depth_matrix, csv_fname, row.names=FALSE, sep=";")